<a href="https://colab.research.google.com/github/mkolennikova/TEB-MSU/blob/main/TEB_sandbox.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TEB-MSU Sandbox

A single notebook that downloads, compiles and runs the TEB-MSU model for a user-defined
location using ERA5 forcing. It works both in **Google Colab** and **locally**
(Windows / Linux / macOS) and lets you choose which source tree to build:
`src_dev` (development physics) or `src_ctrl` (control physics).

Pipeline:
- [Set up environment](#set-up-environment)
- [Download model from repository](#download-model-from-repository)
- [Model compilation](#model-compilation)
- [Download atmospheric forcing](#download-atmospheric-forcing)
- [Configure and run simulations](#configure-and-run-simulations)
- [Preview simulation results](#preview-simulation-results)

## Set up environment

The dictionary `opts` defines everything the notebook needs. Adjust it, then run the cells
top to bottom.

- `src_dir` - source tree to build: **`'src_dev'`** (development physics) or **`'src_ctrl'`**
  (control physics). The executable is detected automatically in `build/`
  (`TEB_offline_dev.exe` / `TEB_offline_ctrl.exe`).
- `repo_path`, `reload_repo` - repository to clone and whether to re-clone it from scratch.
- `work_dir` - working directory for the site data; `None` uses a platform default
  (Google Drive or `/content` on Colab, `<home>/TEB_work` locally).
- `use_gdrive` - Colab only: keep the working directory on Google Drive.
- `toolchain_dirs` - Windows: directories prepended to `PATH`, so that `make` finds the POSIX
  shell, `make` and `gfortran` (MSYS2 defaults: `C:\msys64\usr\bin`, `C:\msys64\ucrt64\bin`,
  `C:\msys64\mingw64\bin`).

The repository is located **automatically**: if the notebook is opened inside an existing
clone (recognised by a source tree, `src_dev/`/`src_ctrl/`, together with the supporting
`python/` or `tests/`), that clone is used as is; otherwise `repo_path` is cloned into
`work_dir`.

The site and the experiment are chosen later, in the *Configure your custom simulation* cell.

### Prerequisites

| Component | How to prepare |
| --- | --- |
| Python kernel (local) | any Python 3 environment; select its kernel in VS Code |
| Python packages (local) | `pip install f90nml cdsapi netCDF4 xarray pandas numpy matplotlib plotly tqdm nbformat` |
| Fortran compiler | Windows: `gfortran` from MSYS2 (`C:\msys64\ucrt64\bin`); Linux: `apt install gfortran` |
| Build tool | Windows: GNU `make` from MSYS2 (`pacman -S --needed make mingw-w64-ucrt-x86_64-gcc`) |
| POSIX shell | Windows: `sh.exe` from the toolchain (MSYS2 by default: `C:\msys64\usr\bin`) - added to `PATH` later |
| ERA5 access | `~/.cdsapirc` (`%USERPROFILE%\.cdsapirc`), created once via `install_utils.init_CDS()` |

On Colab the notebook installs the missing Python packages and uses the toolchain of the
runtime image (`git`, `make`, `gfortran`). Locally the notebook installs nothing - the
environment must be prepared in advance.

In [2]:
import os, sys, glob, time, shutil, subprocess, importlib, importlib.util
import platform
from pathlib import Path

# Works both in Google Colab and in a local Jupyter / VS Code kernel.
try:
    import google.colab  # noqa: F401
    IS_COLAB = True
except ImportError:
    IS_COLAB = False

print(f'Environment: {"Google Colab" if IS_COLAB else "local"} ({platform.system()})')


def import_and_reload(module_name: str):
    """Import a helper module of the repository, or reload it if already imported."""
    if module_name in sys.modules:
        print(f'Reloading existing module: {module_name}')
        return importlib.reload(sys.modules[module_name])
    print(f'Loading new module: {module_name}')
    return importlib.import_module(module_name)

Environment: local (Windows)


In [3]:
opts = {
    # ---- Source tree to build -----------------------------------------------
    'src_dir': 'src_dev',     # 'src_dev' (development physics) or 'src_ctrl' (control physics)

    # ---- Repository ---------------------------------------------------------
    'repo_path': 'https://github.com/mkolennikova/TEB-MSU',
    'reload_repo': False,     # True: delete a cloned copy and clone it again

    # ---- Working directory and platform extras ------------------------------
    'work_dir': None,         # None -> platform default (resolved in the next cell)
    'use_gdrive': False,      # Colab only: keep the working directory on Google Drive

    # Windows: directories prepended to PATH (POSIX shell, make, gfortran).
    'toolchain_dirs': [r'C:\msys64\usr\bin',     # POSIX shell
                       r'C:\msys64\ucrt64\bin',  # make, gfortran
                       r'C:\msys64\mingw64\bin'],
}

In [4]:
MODEL_NAME = os.path.basename(os.path.normpath(opts['repo_path'].rstrip('/')))

# ---- Working directory: a single definition for Colab and local ---------------
# opts['work_dir'] (if set) wins; otherwise a platform default is used:
#   Colab + use_gdrive -> /content/drive/MyDrive/TEB_work
#   Colab              -> /content/TEB_work
#   local              -> <home>/TEB_work
if opts['work_dir']:
    WORK_DIR = Path(opts['work_dir']).expanduser()
elif IS_COLAB:
    if opts['use_gdrive']:
        from google.colab import drive
        drive.mount('/content/drive')
        WORK_DIR = Path('/content/drive/MyDrive/TEB_work')
    else:
        WORK_DIR = Path('/content/TEB_work')
else:
    WORK_DIR = Path.home() / 'TEB_work'
WORK_DIR.mkdir(parents=True, exist_ok=True)


# ---- Model repository: reuse a local clone if the notebook sits in one ---------
def find_repo_root(*start_dirs):
    """Directory of a TEB-MSU clone found at or above `start_dirs`, or None.

    A clone is recognised by one of the source trees (`src_dev/`, `src_ctrl/` or the
    older `src/`) together with a supporting directory (`python/`, `python_tests/` or
    `tests/`).
    """
    for start in start_dirs:
        if not start:
            continue
        start = Path(start).resolve()
        for candidate in [start, *start.parents]:
            has_tree = any((candidate / m).is_dir() for m in ('src_dev', 'src_ctrl', 'src'))
            has_support = any((candidate / m).is_dir() for m in ('python', 'python_tests', 'tests'))
            if has_tree and has_support:
                return candidate
    return None


# VS Code injects `__vsc_ipynb_file__`; Jupyter kernels start in the notebook folder.
NOTEBOOK_DIR = (Path(globals()['__vsc_ipynb_file__']).parent
                if '__vsc_ipynb_file__' in globals() else Path.cwd())
LOCAL_REPO = find_repo_root(NOTEBOOK_DIR, Path.cwd())
MODEL_DIR = LOCAL_REPO if LOCAL_REPO else WORK_DIR / MODEL_NAME
SRC_DIR = MODEL_DIR / opts['src_dir']              # the tree selected by opts['src_dir']
CAPITOUL_DIR = MODEL_DIR / 'tests' / 'CAPITOUL'    # reference test case

# ---- Windows: put the toolchain on PATH (no-op elsewhere) --------------------
# GNU make needs a POSIX shell for `$(shell mkdir -p ...)`, `rm -rf` and `find`, and
# gfortran_args detects the compiler with `command -v ifort`. The directories listed in
# opts['toolchain_dirs'] are prepended so that make finds both the shell and the compiler
# (the MSYS2 default: usr\bin for the shell, ucrt64\bin for make and gfortran).
if not IS_COLAB and platform.system() == 'Windows':
    toolchain_dirs = [Path(d) for d in opts['toolchain_dirs'] if Path(d).is_dir()]
    os.environ['PATH'] = os.pathsep.join(
        [str(d) for d in toolchain_dirs] + [os.environ['PATH']])

print(f'WORK_DIR   : {WORK_DIR}')
print(f'MODEL_DIR  : {MODEL_DIR}' + ('  (existing local clone)' if LOCAL_REPO else '  (will be cloned)'))
print(f'SRC_DIR    : {SRC_DIR}  ({opts["src_dir"]})')
print(f'CAPITOUL   : {CAPITOUL_DIR}')

# ---- Sanity checks -----------------------------------------------------------
if opts['src_dir'] not in ('src_dev', 'src_ctrl'):
    print(f'⚠️ opts["src_dir"] = {opts["src_dir"]!r}: expected "src_dev" or "src_ctrl"')

tool_paths = {tool: shutil.which(tool) for tool in ('git', 'make', 'gfortran', 'sh')}
for tool, path in tool_paths.items():
    print(f'{tool:<9}: {path or "NOT FOUND"}')

missing_tools = [tool for tool, path in tool_paths.items() if path is None]
if missing_tools:
    print('\n❌ Missing tools: ' + ', '.join(missing_tools))
    if IS_COLAB:
        print('   Install them with:  !apt-get update && apt-get install -y gfortran make git')
    else:
        print('   Windows: install a POSIX-like toolchain (e.g. MSYS2, https://www.msys2.org/), then in its shell:')
        print('     pacman -S --needed make mingw-w64-ucrt-x86_64-gcc git')
        print('   and set opts["toolchain_dirs"] to the bin folders of your toolchain.')
else:
    print('✅ Command-line toolchain is ready.')

WORK_DIR   : C:\Users\mvar9\TEB_work
MODEL_DIR  : D:\Work\git\TEB-MSU  (existing local clone)
SRC_DIR    : D:\Work\git\TEB-MSU\src_dev  (src_dev)
CAPITOUL   : D:\Work\git\TEB-MSU\tests\CAPITOUL
git      : C:\Program Files\Git\cmd\git.EXE
make     : C:\msys64\ucrt64\bin\make.EXE
gfortran : C:\msys64\ucrt64\bin\gfortran.EXE
sh       : C:\msys64\usr\bin\sh.EXE
✅ Command-line toolchain is ready.


## Download model from repository

If the notebook is opened inside an existing clone it is used as is (no `git clone`, no
`git pull`). Otherwise the repository is cloned into `work_dir`, or updated with `git pull`
when `reload_repo = False`.

The Colab magics (`!git`, `%rm`) are replaced by `subprocess` / `shutil`, so the same cell
works in a local kernel. The helper modules in `python/` are added to `sys.path`.

In [ ]:
if LOCAL_REPO:
    print(f'Using the existing local clone: {MODEL_DIR} (no git clone / git pull)')
elif not MODEL_DIR.is_dir() or opts['reload_repo']:
    if MODEL_DIR.exists():
        print(f'Removing the previous clone: {MODEL_DIR}')
        shutil.rmtree(MODEL_DIR)
    print(f'Cloning {opts["repo_path"]} -> {MODEL_DIR}')
    subprocess.run(['git', 'clone', opts['repo_path'], str(MODEL_DIR)], check=True)
else:
    print(f'Updating the existing clone: {MODEL_DIR}')
    subprocess.run(['git', '-C', str(MODEL_DIR), 'pull'], check=True)

if not (SRC_DIR / 'Makefile').is_file():
    raise FileNotFoundError(f'{SRC_DIR} has no Makefile - check opts["src_dir"] '
                            f'({opts["src_dir"]!r}) or the repository layout.')

# The helper modules live in python/ 
PYTHON_DIR = MODEL_DIR / 'python'
if not PYTHON_DIR.is_dir():
    raise FileNotFoundError(f'No helper modules found in {MODEL_DIR} (expected python/).')
if str(PYTHON_DIR) not in sys.path:
    sys.path.append(str(PYTHON_DIR))

print(f'✅ Model repository is ready: {MODEL_DIR}')
print(f'✅ Helper modules added to sys.path: {PYTHON_DIR}')

Using the existing local clone: D:\Work\git\TEB-MSU (no git clone / git pull)
✅ Model repository is ready: D:\Work\git\TEB-MSU
✅ Helper modules added to sys.path: D:\Work\git\TEB-MSU\python


### Python libraries

Colab installs the missing packages automatically (`pip`). Locally nothing is installed: the
packages are prepared in advance in the active Python environment (`pip install ...`) and
this cell only verifies that they are importable.

In [6]:
import importlib.util

REQUIRED_MODULES = ['f90nml', 'cdsapi', 'netCDF4', 'xarray', 'pandas',
                    'numpy', 'matplotlib', 'plotly', 'tqdm', 'IPython', 'nbformat']

if IS_COLAB:
    # Colab: `f90nml` and `cdsapi` are not preinstalled - install whatever is missing.
    install_utils = import_and_reload('install_utils')
    install_utils.import_extrernal_modules(REQUIRED_MODULES)
    print('✅ Python libraries are ready (Colab).')
else:
    # Local: nothing is installed - only the presence of the packages is checked.
    missing = [name for name in REQUIRED_MODULES if importlib.util.find_spec(name) is None]
    if missing:
        raise RuntimeError(
            'Missing Python modules: ' + ', '.join(missing) + '\n'
            'This notebook does not install anything locally - prepare the environment first:\n'
            '  pip install ' + ' '.join(missing))
    print('✅ Python libraries are ready (local).')

✅ Python libraries are ready (local).


## Model compilation

The tree selected by `opts['src_dir']` is built with its own `Makefile`
(`make -C <src_dir> clean all`). Sources stay in the tree; the object files and the executable
go to the repository-level `build/` (`obj_dev` / `obj_ctrl`). The notebook then looks for the
newly created executable there.

The `make` logs are written to `build/logs/make_<target>_output.txt`; the whole `build/`
directory (objects, executable, logs) is git-ignored.

Runs are started in the directory of the case, because the driver resolves `namelist/` and the
forcing directory relative to the working directory. The reference test case lives in
`tests/CAPITOUL/` (see the next cell).

Use `['all']` instead of `['clean', 'all']` for an incremental rebuild, and `make_variables`
to append command-line variables to `make` (e.g. to override compiler flags).

In [7]:
make_commands = ['clean', 'all']     # ['all'] = incremental rebuild
make_variables = []                  # e.g. ['OTHERFLAGS=-ffpe-trap=none']

run_utils = import_and_reload('run_utils')

# Build output and logs stay under build/ (git-ignored), not in the repository root.
build_dir = MODEL_DIR / 'build'
log_dir = build_dir / 'logs'
log_dir.mkdir(parents=True, exist_ok=True)

start_time = time.time()
make_return_codes = {}
for command in make_commands:
    make_return_codes[command] = run_utils.run_in_cell(
        exe_path='make',
        args=['-C', str(SRC_DIR)] + make_variables + [command],
        num_lines=10,
        log_file=str(log_dir / f'make_{command}_output.txt'))

new_files = [f for f in glob.glob(os.path.join(str(build_dir), '*.*'))
             if os.path.getmtime(f) >= start_time]
print('\nCompilation finished, new files created:\n' + '\n'.join('\t' + f for f in new_files))

exe_files = [f for f in new_files if f.lower().endswith('.exe')]
EXE_PATH = exe_files[0] if len(exe_files) == 1 else None
if EXE_PATH:
    print(f'✅ Compilation is successful, executable is {EXE_PATH}')
elif not exe_files:
    print('❌ Something is wrong: there are no new executable files')
else:
    print('❌ Something is wrong: more than one new executable file: ' + ', '.join(exe_files))
if EXE_PATH is None:
    for command, rc in make_return_codes.items():
        print(f'   {log_dir / f"make_{command}_output.txt"} (return code {rc})')

gfortran   -funderscoring -O0 -g -Wall -c -fmessage-length=0 -fdefault-real-8 -ffpe-trap=invalid,zero -ffree-line-length-0 -Wno-tabs -Wno-unused -Wno-align-commons -I. -Isrc_teb -J../build/obj_dev -I. -Isrc_teb  -c src_driver/add_forecast_to_date_surf.F90 -o ../build/obj_dev/add_forecast_to_date_surf.o
gfortran   -funderscoring -O0 -g -Wall -c -fmessage-length=0 -fdefault-real-8 -ffpe-trap=invalid,zero -ffree-line-length-0 -Wno-tabs -Wno-unused -Wno-align-commons -I. -Isrc_teb -J../build/obj_dev -I. -Isrc_teb  -c src_teb/bld_occ_calendar.F90 -o ../build/obj_dev/bld_occ_calendar.o
gfortran   -funderscoring -O0 -g -Wall -c -fmessage-length=0 -fdefault-real-8 -ffpe-trap=invalid,zero -ffree-line-length-0 -Wno-tabs -Wno-unused -Wno-align-commons -I. -Isrc_teb -J../build/obj_dev -I. -Isrc_teb  -c src_teb/bld_e_budget.F90 -o ../build/obj_dev/bld_e_budget.o
src_teb/bld_e_budget.F90:7:30:
    7 |                              PRHOA, PT_ROOF, PT_WALL, PTI_BLD, PTS_FLOOR )
      |                 

### Test model run

Runs the reference test case **CAPITOUL** - the Toulouse (France) urban measurement campaign -
with the namelists and the atmospheric forcing bundled in the repository:

* `tests/CAPITOUL/namelist/` - `namelist.nml` (model parameters) and `namelist_forcing.nml`
  (start date, `forc_step`, `nsteps` and the forcing path);
* `tests/CAPITOUL/input/` - the ASCII forcing `Forc_*.txt` of the campaign;
* `tests/CAPITOUL/output_ref/` - the reference output of the case.

The case is a one-year integration starting on 2004-02-20 with `forc_step = 1800 s`. The
executable (absolute path) is started in `tests/CAPITOUL/`, so the driver picks up `namelist/`
and `input/` there with no command-line arguments and writes its own `output/` next to them.
A successful run ends with `TEB-MSU OFFLINE SIMULATION ENDS CORRECTLY`.

In [8]:
if EXE_PATH is None:
    raise RuntimeError('The model executable was not built - fix the compilation step first.')

if not (CAPITOUL_DIR / 'namelist').is_dir():
    raise FileNotFoundError(f'{CAPITOUL_DIR} does not contain the CAPITOUL test case (no namelist/).')

# The driver opens <output_dir>/TEB_output.csv (STATUS='REPLACE'): the directory must exist.
(CAPITOUL_DIR / 'output').mkdir(parents=True, exist_ok=True)

run_utils = import_and_reload('run_utils')
run_utils.run_in_cell(exe_path=EXE_PATH,
                      cwd=str(CAPITOUL_DIR),
                      log_file=str(CAPITOUL_DIR / 'TEB_log.txt'),
                      num_lines=5)

  
     --------------------------
     |  TEB-MSU OFFLINE SIMULATION ENDS CORRECTLY |
     --------------------------
  
✅ Process finished (return code 0). Full log saved to 'D:\Work\git\TEB-MSU\tests\CAPITOUL\TEB_log.txt'.


0

## Download atmospheric forcing

The cells below download ERA5 reanalysis for the configured location and period and convert it
to the ASCII forcing (`Forc_*.txt`) required by the model.

ERA5 comes from the Copernicus Climate Data Store and needs a personal API key stored in
`~/.cdsapirc` (`%USERPROFILE%\.cdsapirc` on Windows). Files already downloaded are skipped, so
the cells can be re-run safely; pass `force_reload=True` to download everything again.

In [9]:
cdsapirc_file = Path.home() / '.cdsapirc'
if cdsapirc_file.is_file():
    print(f'✅ CDS API configuration found: {cdsapirc_file}')
    print('   Delete this file and re-run this cell to enter a new API key.')
else:
    install_utils = import_and_reload('install_utils')
    print(f'⚠️ CDS API configuration not found at {cdsapirc_file}')
    install_utils.init_CDS()
    print(f'✅ CDS API configuration written to {cdsapirc_file}')

✅ CDS API configuration found: C:\Users\mvar9\.cdsapirc
   Delete this file and re-run this cell to enter a new API key.


In [ ]:
forcing_ERA5 = import_and_reload('forcing_ERA5')

# Site folder for the downloaded forcing; reused as `site_name` in the simulation below.
forcing_params = dict(lat = 55.7,
                      lon = 37.5,
                      base_dir = str(WORK_DIR / 'Moscow'),   # absolute -> independent of cwd
                      start_date = '2022-08-01',
                      end_date = '2022-09-01',
                      time = '1h')

# Set chunk_by = 'month' (or 'year') to split long periods into several requests.
# That path uses xarray.open_mfdataset(parallel=True) and requires `dask`.
#forcing_params['chunk_by'] = 'month'

forcing_ERA5.prepare_forcing(**forcing_params)

Loading new module: forcing_ERA5
📥 Downloading ERA5 data for (55.7, 37.5) from 2022-08-01 to 2022-09-01...


2026-10-05 09:44:41,939 INFO Request ID is 1462bc62-efd6-47df-8894-8e6a5b57c5f5
2026-10-05 09:44:42,046 INFO status has been updated to accepted
2026-10-05 09:44:56,903 INFO status has been updated to running


## Configure and run simulations

### Configure your custom simulation based on downloaded forcing

Set the site and the experiment name, then the cell reads the default parameter namelist,
applies your modifications (urban canopy parameters, model options, ...), writes
`params_<exp_name>.nml` and creates the output directory. Site data live under
`<work_dir>/<site_name>/`, so different sites and experiments do not overwrite each other.
`site_name` must match the site folder used for the forcing above.

In [ ]:
import f90nml

site_name = 'Moscow'      # must match the forcing folder used above (`base_dir`)
exp_name = 'CTRL'

SITE_DIR = WORK_DIR / site_name
FORCING_NML_PATH = SITE_DIR / 'forcing_ERA5' / 'namelist_forcing.nml'
PARAMS_NML_PATH = SITE_DIR / f'params_{exp_name}.nml'
OUTPUT_DIR = SITE_DIR / f'output_{exp_name}'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Start from the reference model parameters of the CAPITOUL test case.
params_nml = f90nml.read(str(CAPITOUL_DIR / 'namelist' / 'namelist.nml'))
params_nml['tebparam']['teb_utc_hour'] = 3

# ***** Insert other namelist modifications here *****
# params_nml['tebparam']['...'] = ...

params_nml.write(str(PARAMS_NML_PATH), force=True)

print(f'Forcing    namelist: {FORCING_NML_PATH}')
print(f'Parameters namelist: {PARAMS_NML_PATH}')
print(f'Output directory   : {OUTPUT_DIR}')

### Run model for your custom simulation using command-line arguments

The offline driver takes only a few options - everything else comes from the two namelists.

* **Default run** (no arguments): `namelist/namelist_forcing.nml`, `namelist/namelist.nml` and
  `output/`, relative to the working directory - this is what the *Test model run* cell above
  does in `tests/CAPITOUL/`.
* **`-forcing_nml <file>`** - use another forcing namelist (`/tebforcing/`: start date,
  `forc_step`, `nsteps`, point coordinates and `forcing_path`).
* **`-param_nml <file>`** - use another parameter namelist (`/tebparam/`: urban canopy, BEM,
  garden and greenroof options).
* **`-output <dir>`** - write `<dir>/TEB_output.csv`; the folder must exist, an existing file is
  overwritten.
* **`-h` / `-help` / `--help`** - print the usage summary and exit.

The options can be combined and given in any order; an unknown option stops the run. Relative
paths are resolved against the working directory, so the driver is started with `cwd` set to the
case directory.

The custom run below passes all three values with absolute paths.

In [ ]:
if EXE_PATH is None:
    raise RuntimeError('The model executable was not built - fix the compilation step first.')

run_utils = import_and_reload('run_utils')
run_utils.run_in_cell(exe_path=EXE_PATH,
                      args=['-forcing_nml', str(FORCING_NML_PATH),
                            '-param_nml', str(PARAMS_NML_PATH),
                            '-output', str(OUTPUT_DIR)],
                      log_file=str(OUTPUT_DIR / 'model_output.log'),
                      cwd=str(MODEL_DIR), num_lines=5)

In [ ]:
output_utils = import_and_reload('output_utils')

# read_output takes the output file itself (or its directory) and keeps the 'Forc_*' columns
# written by the model - they are the forcing overlay of the preview cells below.
OUTPUT_CSV = OUTPUT_DIR / 'TEB_output.csv'
output_df = output_utils.read_output(OUTPUT_CSV)
print(f'{len(output_df)} output time steps')
print('variables: ' + ', '.join(output_df.columns))

### Preview simulation results

Static (Matplotlib) and interactive (Plotly) plots of the run. The atmospheric forcing is
overlaid as **black reference lines**: air temperature at the forcing level on the
*Temperatures* panel and wind speed at the forcing level on the *Wind Speed* panel.

The overlay is taken from the model output itself: `read_output()`, given the output directory
or the `TEB_output.csv` path, keeps the `Forc_*` columns that the model writes for every step,
so a separate forcing file is not needed. For older
outputs without those columns the forcing can still be passed explicitly, e.g.
`preview_output_mpl(output_df, forcing_df=forcing_utils.read_forcing(nml_path))`.

The Matplotlib figure is also saved as a PNG next to the model output.

In [ ]:
fig_mpl, axes_mpl = output_utils.preview_output_mpl(
    output_df,
    save_path=str(OUTPUT_DIR / 'preview_output_mpl.png'))

In [ ]:
output_utils.preview_output_plotly(output_df)

## Notes and troubleshooting

- **Choosing the source tree.** `opts['src_dir']` selects `'src_dev'` or `'src_ctrl'`; the
  notebook builds it with `make -C <src_dir>` and detects the executable in `build/`.
- **Colab vs local.** `IS_COLAB` is detected automatically: Colab installs the Python packages
  and clones the repository, while locally the notebook installs nothing and reuses a local
  clone automatically (when the notebook is opened inside one).
- **Local kernel.** Select the kernel of your prepared Python environment (Select Kernel ->
  Python Environments).
- **Plotly preview.** The interactive figure is rendered through the Jupyter mime machinery,
  so the kernel environment needs `nbformat` (`pip install nbformat`; already present on
  Colab).
- **Forcing overlay.** The preview plots take the `Forc_*` columns from the model output
  itself (the model writes the forcing of every step into `TEB_output.csv`);
  `forcing_utils.read_forcing()` is only needed for older runs that do not write them.
- **POSIX-like toolchain for make.** `make` needs a POSIX shell and utilities; on Windows the bin
  directories of `opts['toolchain_dirs']` (MSYS2 defaults) are prepended to `PATH`, otherwise
  the `Makefile` constructs (`$(shell mkdir -p ...)`, `rm -rf`, `find`) and the compiler
  detection in `gfortran_args` (`command -v ifort`) fail.
- **No shell in the notebook.** Every program is started through an argument list
  (`subprocess` without `shell=True`), so the notebook never invokes `sh` itself.
- **Rebuilding.** `make_commands = ['clean', 'all']` rebuilds from scratch; use `['all']` for an
  incremental build. Compiler-flag changes require `clean`, because `make` does not track flags.
- **`make_variables`** appends variables to `make`, e.g. `['OTHERFLAGS=-ffpe-trap=none']` to
  relax the strict IEEE traps of `gfortran_args`.
- **Paths with spaces** are safe: the executable is started through an argument list, not a shell.
- **Relative paths of the model.** The driver resolves `namelist/`, the forcing directory and
  `output/` relative to the working directory, so every run is started in the directory of its
  case (the reference case is `tests/CAPITOUL/`).
- **Reference test case.** CAPITOUL (Toulouse, France) is the reference case shipped in
  `tests/CAPITOUL/` - namelists, ASCII forcing (`input/`) and the reference output
  (`output_ref/`); the *Test model run* cell runs it in place.
- **Long ERA5 requests.** Set `chunk_by='month'` (or `'year'`) in `forcing_params`; that path
  requires `dask`.
- **CDS key.** Stored in `~/.cdsapirc` and written by `install_utils.init_CDS()`; the notebook
  asks for it only when the file is missing.
- **Source fixes.** The reproducibility fixes of the local (Windows) run are described in
  `docs/TEB_MSU_source_defects.md`.